<a href="https://colab.research.google.com/github/rishyg20/Alpha-Research---Assignment-2/blob/main/Alpha_Research_Assignment2_Pushkal_Rishabh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alpha Research — Assignment 2
### Team: Pushkal & Rishabh

In [2]:
!pip install fredapi
from fredapi import Fred
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from datetime import datetime, timedelta
import time
import os
import requests

from google.colab import drive

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/AlphaResearch_Assignment2/data/raw/'

Mounted at /content/drive


## Section 2: S&P 500 Constituents

In [ ]:
start = time.time()
header = {'User-Agent': 'Chrome'}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(url, headers= header)
tables = pd.read_html(response.text)
sp500 = tables[0]

today = datetime.today().strftime("%Y-%m-%d")
sp500['download_date'] = today
print(sp500.shape)
print(sp500.head())
display(sp500)
os.makedirs(DATA_PATH, exist_ok = True)
sp500.to_csv(DATA_PATH + f"sp500_constituents_{today}.csv", index = False)
end = time.time()
print(f"Time taken: {round(end - start,2)} seconds")

In [ ]:
start = time.time()

stock_list = sp500['Symbol'].tolist()
end_date = datetime.today().strftime("%Y-%m-%d")
start_date = "2016-01-01"
#downloading Adj Stock price of SP500 constituents
prices = yf.download(stock_list, start = start_date, end = end_date, auto_adjust = True)
display(prices)
print(prices.shape)


end = time.time()
print(f"Time taken: {round(end - start,2)} seconds")
#yfinance drops Adj Close when auto_adjust=True — adjusted prices are stored under Close instead.

In [ ]:
start = time.time()
adj_close = prices['Close']
print(adj_close.shape)
print(adj_close.head())
adj_close.to_csv("DATA_PATH" + f"prices_close{today}.csv")
print(f"File saved for {adj_close.shape[1]} stocks, {adj_close.shape[0]} trading days")

end = time.time()
print(f"Time taken: {round(end - start,2)} seconds")

In [ ]:
#converting stock prices to monthly returns
start = time.time()
monthly_prices = adj_close.resample('ME').last()
monthly_returns = monthly_prices.pct_change()
print(monthly_returns.head())
print(monthly_returns.shape)
end = time.time()
print(f"Time taken: {round(end - start,2)} seconds")

In [ ]:
#Feature creation - Momentum & Volatility

start = time.time()

momentum = monthly_returns.shift(1).rolling(window=11).sum()
print(momentum.shape)
print(momentum.head(15))
#Feature 2 : Volatility
volatility = monthly_returns.rolling(window = 12).std()
print(volatility.shape)

vix = yf.download("^VIX", start = start_date, end = end_date, auto_adjust = True)['Close']
vix_m = vix.resample("ME").last()
print(vix_m.shape)
end = time.time()
print(f"Time taken: {round(end - start,2)} seconds")

In [ ]:
#Macro data (FRED)



FRED_API_KEY = "4ebdf40a69f480d4b2d0969197ec30f6"
fred_api = Fred(api_key = FRED_API_KEY)

SERIES = {
    'DGS3MO'      : '3 Month Treasury Yield',
    'T10Y2Y'      : 'Yield Curve 10yr-2yr',
    'DTWEXBGS'    : 'Trade Weighted Dollar Index',
    'DCOILWTICO'  : 'WTI Crude Oil Price',
    'BAMLH0A0HYM2': 'Credit Spread',
    'UNRATE'      : 'Unemployment Rate',
    'CPIAUCSL'    : 'Inflation CPI',
    'GDPC1'       : 'Real GDP',
    'M2SL'        : 'M2 Money Supply',
    'PCECC96'     : 'Real Personal Consumption',
}

#Downloading each series and storing it in a dict
fred_dict = {}
for series_id, name in SERIES.items():
  fred_dict[series_id] = fred_api.get_series(series_id, observation_start = start_date, observation_end = end_date)
  print(f"Downloaded: {name}: {len(fred_dict[series_id])} observations")


fred_df = pd.DataFrame(fred_dict)

#Resampling to monthly

fred_monthly = fred_df.resample("ME").last().ffill()

#Saving to Drive
os.makedirs(DATA_PATH, exist_ok = True)
print(f"\nFRED data shape: {fred_monthly.shape}")
display(fred_monthly.head(10))
print(fred_monthly.isnull().sum())
fred_monthly= fred_monthly.drop(columns = ['BAMLH0A0HYM2']) #dropping credit spread data since 88 missing values
print(fred_monthly.isnull().sum())
print(f"\nFRED data shape: {fred_monthly.shape}")
fred_monthly.to_csv(DATA_PATH + f'fred_macro_{today}.csv')
end = time.time()
print(f"Time taken: {round(end - start,2)} seconds")

In [ ]:
# DATA QUALITY OBSERVATIONS - FRED Macro Data
# =============================================

# DGS3MO (3M Treasury): 2706 daily obs → resampled to 125 monthly
# T10Y2Y (Yield Curve): 2707 daily obs → resampled to 125 monthly
# DTWEXBGS (Dollar Index): 2706 daily obs → resampled to 125 monthly
# DCOILWTICO (WTI Oil): 2702 daily obs → resampled to 125 monthly

# BAMLH0A0HYM2 (Credit Spread): 88 missing values at start of period
# → Series begins after 2016-01-01, shorter history than other series
# → Will handle with dropna() or ffill() during join

# UNRATE, CPIAUCSL, M2SL: Monthly frequency, 124 obs
# GDPC1, PCECC96: Quarterly frequency, only 41 obs
# → Forward filled to monthly using .ffill()

# OVERALL: 125 rows x 10 columns after resampling
# Matches monthly_returns shape (125 x 503) - ready for joining

In [ ]:
#Section 4b - Fama French
import zipfile
import io

start = time.time()

#url from Kenneth French Data Library
ff_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip'

#download and unzip
r = requests.get(ff_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
ff_raw = pd.read_csv(z.open(z.namelist()[0]), skiprows = 3, index_col = 0)

print(ff_raw.head(10))
print(ff_raw.tail(10))
print(ff_raw.index[:5].tolist())

#Keeping monthly values only
def is_valid_month(val):
  try:
    v = int(str(val).strip())
    return 190000 <= v <= 210000
  except:
    return False

ff_monthly = ff_raw[ff_raw.index.map(is_valid_month)].copy()
print(ff_monthly.index[:10].tolist())  # show first 10 index values
print(type(ff_monthly.index[0]))       # show what type they are
ff_monthly.index = pd.to_datetime(ff_monthly.index.astype(str), format = "%Y%m")
ff_monthly.index = ff_monthly.index + pd.offsets.MonthEnd(0)

ff_monthly = ff_monthly.apply(pd.to_numeric, errors = 'coerce')/100

ff_monthly = ff_monthly[start_date:end_date]

ff_monthly.to_csv(DATA_PATH + f"fama_french_5factors_{today}.csv")

print(f"Fama - French shape: {ff_monthly.shape}")
print(f"Columns: {ff_monthly.columns.tolist()}")

display(ff_monthly.head())

end = time.time()



print(f"Time taken: {round(end - start,2)} seconds")



In [ ]:
# Joining the data

start = time.time()
# Stack each wide dataframe into long format
returns_long = monthly_returns.stack().reset_index()
returns_long.columns = ['date', 'ticker', 'return']

print(returns_long.shape)
display(returns_long.head())

momentum_long = momentum.stack().reset_index()
momentum_long.columns = ['date', 'ticker', 'momentum']

print(momentum_long.shape)
display(momentum_long.head())

volatility_long = volatility.stack().reset_index()
volatility_long.columns = ['date', 'ticker', 'volatility']

print(volatility_long.shape)
display(volatility_long.head())

#Merging

panel = returns_long.merge(momentum_long, on = ['date', 'ticker'], how = 'inner')
panel = panel.merge(volatility_long, on = ['date', 'ticker'], how = 'inner')

print(panel.shape)
display(panel.head())


end = time.time()



print(f"Time taken: {round(end - start,2)} seconds")
